In [1]:
import os
import HeST as hest
import HeST.Amherst_split_cpd as examp
import numpy as np
import matplotlib.pyplot as plt
import HeST.Detection as detection
%matplotlib qt

import astropy.stats as astat
from scipy.interpolate import interp1d

In [2]:

detector = examp.Amherst_split_cpd 


QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )

QP_conditions.append( detector.liquid_surface )
detector.set_QP_reflection_prob(0.80)
detector.set_diffuse_prob(0.0)
detector.set_evaporation_eff(1.0)

In [3]:
def plot_stacked_hist(evap):
    cpd_1_ints = np.unique(evap.bounce_flag[0])

    cpd_2_ints = np.unique(evap.bounce_flag[1])
    masks_cpd_1 = np.empty((len(cpd_1_ints), len(evap.bounce_flag[0])), dtype = int)
    masks_cpd_2 = np.empty((len(cpd_2_ints), len(evap.bounce_flag[1])), dtype = int)
    cpd1_times = evap.arrivalTimes_us[0]
    cpd2_times = evap.arrivalTimes_us[1]
    plt.figure()
    for ii, bounce_num in enumerate(cpd_1_ints):
        mask = evap.bounce_flag[0] == bounce_num 
        plt.hist(cpd1_times[mask],bins = 200, range = [0,3000], alpha= 0.9, label = 'cpd1, bounce = ' + str(bounce_num - 1))
    plt.title('Arrival times, with bounce')
    plt.xlabel('Time [us]')
    plt.legend()
    plt.show()
    plt.figure()
    for ii, bounce_num in enumerate(cpd_2_ints):
        mask = evap.bounce_flag[1] == bounce_num
        plt.hist(cpd2_times[mask], bins = 200, range = [0.0, 3000.0],alpha = 0.9, label= 'cpd2, bounce = ' + str(bounce_num - 1) )
    plt.title('Arrival Times, with bounce')
    plt.xlabel('Time [us]')
    plt.legend()
    plt.show()

In [5]:
pos = [0.0, 0.0, 1.5]
useMap = False
dir_unnormalized = np.array([0, .3, 1])
dir =dir_unnormalized/np.sqrt(np.sum(dir_unnormalized**2))
evap = hest.GetEvaporationSignal( detector, 1, *pos, useMap=useMap, debug=True, debug_dir=dir,  plot_3d=True, choose_momentum=True, momentum_choice= 4.5, verbose=False)
plt.figure()
plt.hist(evap.arrivalTimes_us[0], bins=200, range=[0, 3000], histtype='step', color='r', lw=2, label = 'CPD1')
plt.hist(evap.arrivalTimes_us[1], bins=200, range=[0, 3000], histtype='step', color='b', lw=2, label = 'CPD2')
plt.legend()
plt.title('Arrival Times')
plt.xlabel('time [us]')
plt.ylabel('Counts/bin')
plt.show()

~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


In [6]:
critical_angles = np.array([-0.99914392,  -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392])

random_numbs = np.array([-0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392, -0.99914392])
print((critical_angles > 1.19028995) & (random_numbs > 1.0))

[False False False False False False False False False False]


In [33]:
def critical_angle(Energy, momentum, binding_energy = 0.00062e-3):
    m =  3.725472e6 #He mass in keV/c^2
    c = 2.998e8
    return np.arcsin(1 * np.sqrt(2 * m * (Energy - binding_energy))/momentum)
def GetInterpFunc(d_path):
    """Creates an linear interpolation function from data found at the file path below,, giving us the ability to convert from resistance to temperature. 
    returns:
        Interpoltion function: If input exceeds range of the data function returns a NaN"""
    data = np.loadtxt(d_path, delimiter=',')
    X = data[:,0]
    Y = data[:,1]
    print(argrelextrema(Y, np.less))
    print(X[101])
    return interp1d(X,Y, kind = 'linear')

def QP_dispersion(p ):
    """generate the energy of the particle in meV

    Args:
        p (int): the momentum of the quasiparticle of interest 
        interp (funct): function to relate momentum and energy  
    """
    interp = GetInterpFunc('./dispersion_data.csv')
    energy = interp(p)
    return energy * 1e-3 #This is in eV 


momentum = np.linspace(0.145, 4.7, 134)
print(QP_dispersion(momentum)*1e6)
# need to print the dispersion curve here too

energy = QP_dispersion(momentum) *1e-3 
plt.figure()
plt.plot(momentum, QP_dispersion(momentum)/np.max(QP_dispersion(momentum)), alpha = 0.3, label = 'normalized Dispersion Curve for reference')
plt.plot(momentum, critical_angle(energy, momentum, binding_energy=0.00062e-3), label = 'Critical Angle')
plt.axvline(3.84)
plt.legend()
plt.xlim(0, 4.7)
plt.xlabel('Momentum [KeV/c]')
plt.ylabel(r'$\theta_c$ in radians')
plt.title('Critical Evaporation Angle')

(array([ 63, 101]),)
3.840354441491354
[  80.85322495  111.89234511  140.60846348  169.67795446  198.97560751
  228.27258895  257.28200103  287.82937609  316.77107351  347.01523869
  376.01914778  406.72791582  435.81484473  465.95103454  495.11563909
  525.57635462  554.49300053  583.69371267  612.38398207  641.99455131
  670.60780194  696.74490846  723.98569686  750.44412024  776.92207852
  802.29655195  827.13436958  851.82505042  878.66302204  902.22255299
  924.23334727  945.17319669  965.04833419  984.44660171 1002.96087778
 1020.22788539 1037.00328262 1052.8170796  1067.58327216 1081.63668506
 1094.79682325 1107.02873872 1118.5518607  1129.58145098 1139.43199183
 1148.91177467 1157.72722939 1165.84963744 1173.38687111 1180.39725571
 1186.87587421 1192.51870475 1198.03948311 1203.08560419 1207.49085193
 1211.43629152 1215.01856698 1218.52210471 1220.80378892 1222.72241412
 1224.64516042 1226.03608595 1225.72577451 1225.1915512  1225.23594208
 1224.13364497 1222.10213949 1219.5011

Text(0.5, 1.0, 'Critical Evaporation Angle')

In [38]:

momentum = np.linspace(0.145, 4.7, 134)
plt.plot(momentum, QP_dispersion(momentum)*1e3)

(array([ 63, 101]),)
3.840354441491354


In [52]:
from scipy.signal import argrelextrema
def get_phonon_mom_energy(d_path):
    data = np.loadtxt(d_path, delimiter=',')
    X = data[0:61,1]
    Y = data[0:61,0]
    return interp1d(X,Y, kind = 'linear')
def get_rminus_mom_energy(d_path):
    data = np.loadtxt(d_path, delimiter=',')
    X = data[61:101,1]
    Y = data[61:101,0]
    return interp1d(X,Y, kind = 'linear')
def get_rplus_mom_energy(d_path):
    data = np.loadtxt(d_path, delimiter=',')
    X = data[101:,1]
    Y = data[101:,0]
    return interp1d(X,Y, kind = 'linear')

phonon_interp = get_phonon_mom_energy('./dispersion_data.csv')
rminus_interp = get_rminus_mom_energy('./dispersion_data.csv')
rplus_interp = get_rplus_mom_energy('./dispersion_data.csv')
energy = np.linspace(764, 1223, 50) * 1e-3
plt.plot(energy, phonon_interp(energy))
plt.plot(energy, rminus_interp(energy))
plt.plot(energy, rplus_interp(energy))
plt.xlabel('energy')
plt.ylabel('Momentum')
plt.title('Inverting dispersion relation in the area of energy degeneracy')


Text(0.5, 1.0, 'Inverting dispersion relation in the area of energy degeneracy')

In [56]:
@np.vectorize
def assign_flavors(p, flavor):
    if .955 < p <   2.168:
        flavor = 'phonon'
    if 2.383 < p < 3.77:
        flavor = 'R-'
    if 3.843<p < 4.541:

        flavor = 'R+'
    return flavor

In [59]:
flavor = np.full(10, 'out_of_degenerate_range')
momentum = np.random.uniform(low = .147, high = 4.7, size=10)
print(assign_flavors(momentum, flavor))

['R-' 'phonon' 'R-' 'out_of_degenerate_range' 'R+'
 'out_of_degenerate_range' 'R-' 'R-' 'R+' 'phonon']


In [19]:
plt.hist(evap.num_bounces, bins=20 )
plt.yscale('log')
plt.xlabel('number of bounces')
plt.ylabel('Counts per num of bounce')
plt.title('Understanding if we are gettign the expected number of bounces')
print(np.mean(evap.num_bounces))

6.767


In [ ]:
pos = [0., 0., 1.5]
useMap = False
evap = hest.GetEvaporationSignal( detector, 10, *pos, useMap=useMap, debug=False, plot_3d=True)


In [ ]:
plot_stacked_hist(evap)


In [24]:
pos = [0., 0., 1.5]
useMap = False
evap = hest.GetEvaporationSignal( detector,1000, *pos, useMap=useMap, debug=False, plot_3d=False)
plot_stacked_hist(evap=evap)


Starting at [0.0, 0.0, 1.5]
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


[[0.  0.  0.  ... 0.  0.  0. ]
 [0.  0.  0.  ... 0.  0.  0. ]
 [1.5 1.5 1.5 ... 1.5 1.5 1.5]]
[[-0.46090506 -0.56086079 -0.01281864 ...  0.35942796  0.65742268
   0.93227148]
 [ 0.04038853 -0.07605929  0.83307274 ...  0.77727326  0.02410964
  -0.29520348]
 [-0.88652991 -0.82440898 -0.55301491 ...  0.51638922  0.7531362
   0.20910472]]
XY [[ True  True  True ... False False False]
 [ True  True  True ... False False False]
 [ True  True  True ... False False False]
 ...
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]]
Z [[ True  True  True ... False False False]
 [ True  True  True ... False False False]
 [ True  True  True ... False False False]
 ...
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]]
Liquid [[ True  True  True ...  True

# Status Update
At this point, everything above this is fully being used to 'flush' out, and as an active debugging site. Below this we will be beginning to run tests of the code, and this way I can save it for the future

In [3]:

detector = examp.Amherst_split_cpd 


QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )

QP_conditions.append( detector.liquid_surface )
detector.set_QP_reflection_prob(0.80)
detector.set_diffuse_prob(0.0)
detector.set_evaporation_eff(1.0)

This code block is dedicated towards making a bunch of plots of the paths, where we are varying the momentum (with the same set of directions)

It is important to note that it saves this is a path _plots/adams-thesis_recreation_ with the name value of momentum + _xy.png_

In [11]:
pos = [0.0, 0.0, 1.5]
moms = np.linspace(1.4, 4.7, 5)
directions_y = np.linspace(2, -2, 9)
paths = []
mom =4.4
for ii, dirs in enumerate(directions_y):
    print('\n')
    print(f'Starting the {ii}th direction')
    useMap = False
    dir_unnormalized = np.array([0, 1, 1])
    dir_unnormalized[1]= dirs
    dir =dir_unnormalized/np.sqrt(np.sum(dir_unnormalized**2))
    print(dir)
    evap = hest.GetEvaporationSignal( detector, 4, *pos, useMap=useMap, debug=True, debug_dir=dir,  plot_3d=True, choose_momentum=True, momentum_choice= mom)
    paths.append(evap.positions)
plt.show()
plt.figure()
for ii, element in enumerate(paths):
    print(element[2][0,:])
    mask = (element[2][0,:] == 0)  & (element[1][0,:] == 0)
    plt.plot(element[1][0,:][~mask], element[2][0,:][~mask], label = (round(directions_y[ii], 2), 1))
plt.axhline(3.3, color ='g', lw = 3, label = 'CPD surface')
plt.axhline(2.75, color = 'b', lw = 3, label = 'liquid surface')
plt.legend(loc = 'lower left')
plt.xlabel('Y-coordinate')
plt.ylabel('Z-coordinate')
plt.title(f'This is with a momentum of {mom}')
plt.savefig(os.path.join('plots', 'adams-thesis_recreation',  str(mom) + 'xy.png'))
plt.show()



Starting the 0th direction
[0.         0.89442719 0.4472136 ]
Starting at [0.0, 0.0, 1.5]
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


[[0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [1.5 1.5 1.5 1.5]]
[[0.         0.         0.         0.        ]
 [0.89442719 0.89442719 0.89442719 0.89442719]
 [0.4472136  0.4472136  0.4472136  0.4472136 ]]
['Liquid' 'Liquid' 'Liquid' 'Liquid']
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
[0.42118421 0.42118421 0.42118421 0.42118421]
this is critical angles [0.43475059 0.43475059 0.43475059 0.43475059] and this is incident [1.10714872 1.10714872 1.10714872 1.10714872]
1.0
this is the randomely generated numbers [0.97439496 0.62916456 0.53279668 0.43618648]
[False False False False]
[ True  True  True  True]
[]
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


[[0.         0.         0.         0.        ]
 [0.96921207 0.96921207 0.96921207 0.96921207]
 [1.88460604 1.88460604 1.88460604 1.88460604]]


now we want to make a similar code block, except where we are varying the height of the liquid. 

In [13]:

detector = examp.Amherst_split_cpd 
pos = [0.0, 0.0, 1.5]
QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )
detector.set_QP_reflection_prob(0.80)
detector.set_diffuse_prob(0.0)
detector.set_evaporation_eff(1.0)

directions_y = np.linspace(-1, 1, 10, dtype=float)
paths = []
h = 3.00
mom = 3.0
for ii, dirs in enumerate(directions_y):
    print('\n')
    print(f'Starting the {ii}th direction')
    useMap = False
    dir_unnormalized = np.array([0, 1, 1], dtype=float)
    dir_unnormalized[1]= dirs
    dir =dir_unnormalized/np.sqrt(np.sum(dir_unnormalized**2))
    evap = hest.GetEvaporationSignal( detector, 4, *pos, useMap=useMap, debug=True, debug_dir=dir,  plot_3d=True, choose_momentum=True, momentum_choice= mom, verbose = False)
    paths.append(evap.positions)
plt.show()
plt.figure()
bounces = evap.bounce_flag
for ii, element in enumerate(paths):
    print(element[2][0,:])
    mask = (element[2][0,0:3] == 0)  & (element[1][0,0:3] == 0)

    plt.plot(element[1][0,0:3][~mask], element[2][0,0:3][~mask], label = (round(directions_y[ii], 2), 1))
plt.axhline(3.3, color ='g', lw = 3, label = 'CPD surface')
plt.axhline(h,  color = 'b', lw = 3, label = 'liquid surface')
plt.legend(loc = 'lower left')
plt.xlabel('Y-coordinate')
plt.ylabel('Z-coordinate')
plt.title(f'This is with a momentum of {mom} and a height of {h}')
plt.savefig(os.path.join('plots', 'height_variance',  str(h)+'_' +str(mom) +  '_xy.png'))
plt.show()
plt.close('all')



Starting the 0th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~


Starting the 1th direction
~~~~~~~~~~~~~~~~ Findi

In [4]:

detector = examp.Amherst_split_cpd 
pos = [0.0, 0.0, 1.5]
QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )
detector.set_QP_reflection_prob(0.80)
detector.set_diffuse_prob(0.0)
detector.set_evaporation_eff(1.0)

directions_y = np.linspace(-1, 1, 10, dtype=float)
momentums = np.array([2.0, 2.5, 3.0, 3.5, 4.0, 4.5])
for mom in momentums:
    paths = []
    print(mom)
    h = 2.0
    for ii, dirs in enumerate(directions_y):
        print('\n')
        print(f'Starting the {ii}th direction')
        useMap = False
        dir_unnormalized = np.array([0, 1, 1], dtype=float)
        dir_unnormalized[1]= dirs
        dir =dir_unnormalized/np.sqrt(np.sum(dir_unnormalized**2))
        evap = hest.GetEvaporationSignal( detector, 4, *pos, useMap=useMap, debug=True, debug_dir=dir,  plot_3d=True, choose_momentum=True, momentum_choice= mom, verbose = False)
        paths.append(evap.positions)
    plt.show()
    plt.figure()
    bounces = evap.bounce_flag
    for ii, element in enumerate(paths):
        print(element[2][0,:])
        mask = (element[2][0,0:3] == 0)  & (element[1][0,0:3] == 0)

        plt.plot(element[1][0,0:3][~mask], element[2][0,0:3][~mask])
    plt.axhline(3.3, color ='g', lw = 3, label = 'CPD surface')
    plt.axhline(h,  color = 'b', lw = 3, label = 'liquid surface')
    plt.legend(loc = 'lower left')
    plt.xlabel('Y-coordinate')
    plt.ylabel('Z-coordinate')
    plt.title(f'This is with a momentum of {mom} and a height of {h}')
    plt.savefig(os.path.join('plots', 'height_variance',  str(h)+'_' +str(mom) +  '_xy.png'))
    plt.show()
    plt.close('all')

2.0


Starting the 0th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


Starting the 1th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


Starting the 2th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


Starting the 3th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~ Computing Evaporations ~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


Starting the 4th direction
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~
~~~

In [14]:
plt.close('all')

In [21]:
dir_unnormalized = np.array([0, 1, 1], dtype = float)
print(dir_unnormalized[1])
directions_y = np.linspace(-1, 1, 10)
for dy in directions_y:
    dir_unnormalized[1] = dy
    print(dy)
    print(dir_unnormalized)
    dir =dir_unnormalized/np.sqrt(np.sum(dir_unnormalized**2))
    print(dir)
 

1.0
-1.0
[ 0. -1.  1.]
[ 0.         -0.70710678  0.70710678]
-0.7777777777777778
[ 0.         -0.77777778  1.        ]
[ 0.         -0.61394061  0.78935222]
-0.5555555555555556
[ 0.         -0.55555556  1.        ]
[ 0.         -0.48564293  0.87415728]
-0.33333333333333337
[ 0.         -0.33333333  1.        ]
[ 0.         -0.31622777  0.9486833 ]
-0.11111111111111116
[ 0.         -0.11111111  1.        ]
[ 0.         -0.11043153  0.99388373]
0.11111111111111116
[0.         0.11111111 1.        ]
[0.         0.11043153 0.99388373]
0.33333333333333326
[0.         0.33333333 1.        ]
[0.         0.31622777 0.9486833 ]
0.5555555555555554
[0.         0.55555556 1.        ]
[0.         0.48564293 0.87415728]
0.7777777777777777
[0.         0.77777778 1.        ]
[0.         0.61394061 0.78935222]
1.0
[0. 1. 1.]
[0.         0.70710678 0.70710678]


In [38]:
# The first thing I want to make is a set of plots that are similar to the ones from the adams thesis.


Starting at [0.0, 0.0, 1.5]
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


[[0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [1.5 1.5 1.5 1.5 1.5 1.5 1.5 1.5 1.5 1.5]]
[[-0.43128571 -0.47298528 -0.65405656  0.49601139 -0.48414863 -0.91244894
   0.32701851 -0.08477746 -0.74606721  0.1910398 ]
 [-0.81243106  0.85105461 -0.07729639 -0.64965263  0.26136373 -0.31041775
   0.56538731  0.62364733 -0.438338   -0.98138039]
 [-0.39236259 -0.22801529 -0.75248607 -0.5761286  -0.83503839  0.26660411
   0.75722921  0.7770951   0.50124198 -0.0199081 ]]
[[ 0.00000000e+00 -3.26059290e-02 -6.52118580e-02 ... -9.68396091e+00
  -9.71656684e+00 -9.74917276e+00]
 [ 0.00000000e+00  3.41560379e-02  6.83120758e-02 ...  1.01443433e+01
   1.01784993e+01  1.02126553e+01]
 [ 0.00000000e+00 -3.10219624e-03 -6.20439248e-03 ... -9.21352284e-01
  -9.24454480e-01 -9.27556676e-01]
 ...
 [ 0.00000000e+00  2.50293244e-02  5.00586488e-02 ...  7.43370935e+00
   7.45873

UnboundLocalError: local variable 'particles_x' referenced before assignment

In [ ]:
n = np.array([[0,1,2],[4,5,6]])
print(n[:,0])

In [ ]:
probs = np.linspace(0, .9, 10)
counts = np.empty_like(probs)
for ii, p in enumerate(probs):   
    pos = [0., 0., 1.5]
    detector.set_QP_reflection_prob(p)
    
    useMap = False
    evap = hest.GetEvaporationSignal( detector, 50000,*pos, useMap=useMap)
    counts[ii] = len(evap.arrivalTimes_us[0])

plt.plot(probs, counts)
plt.title('Logical Check. As the number of reflections goes up, so the should the number of events we have')
plt.legend()



In [ ]:
import HeST.Detection as detection
import os
import numpy as np
#The detector geometry is defined from the point of view of particle paths.
# We essentially want to define various "surface conditions" where the particle paths are obstructed
# These functions also carry a "boundary_type", so that we can keep track if the particle is obstructed by
# a CPD, or a wall, and how it may reflect off of a given wall.

def sensor1_conditions(x, y, z):
    boundary_type = "CPD0"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) & (x>0)& (z < height)| (x*x + y*y >= radius*radius) , boundary_type

def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type





baseline_noise = [0., 0.]
phonon_conversion = 0.25
cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)





def wall_conditions(x, y, z):
    boundary_type = "XY"
    radius = 3. #cm
    height = 2.75 #cm
    return ((x*x + y*y < radius*radius) & (z < height) ) | (z > height), boundary_type

def bottom_conditions(x, y, z):
    boundary_type = "Z"
    bottom = 0. #cm
    return (z > bottom), boundary_type

def liquid_surface(x, y, z):
    boundary_type = "Liquid"
    height = 2.75 #cm
    return (z < height), boundary_type

def liquid_conditions(x, y, z):
    height = 2.75 #cm
    radius = 3. #cm
    bottom = 0. #cm
    return ((x*x + y*y < radius*radius) & (z < height) & (z > bottom))
   

Amherst_split_cpd = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

 

In [ ]:
def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type

detector= detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)


cpd_2_x_bounds = np.linspace(0,-3.1, 10)
signals_cpd1 = np.empty_like(cpd_2_x_bounds)
signals_cpd2 = np.empty_like(cpd_2_x_bounds)


for ii, bound in enumerate(cpd_2_x_bounds):
    pos = [0., 0., 1.5]
    def sensor2_conditions(x, y, z):
        boundary_type = "CPD1"
        radius = 3.8
        height = 3.3
        print(bound)
        return (x*x + y*y < radius*radius) &  (x<bound)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type
    cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
    cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)


    detector = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

    useMap = False
    evap = hest.GetEvaporationSignal( detector, 40000, *pos, useMap=useMap)
    print( evap.area_eV, evap.chArea_eV, evap.coincidence, len(evap.arrivalTimes_us))
    signals_cpd1[ii] = len(evap.arrivalTimes_us[0])
    signals_cpd2[ii] = len(evap.arrivalTimes_us[1])

plt.plot(cpd_2_x_bounds, signals_cpd1, label = 'Without changing size')
plt.plot(cpd_2_x_bounds, signals_cpd2, label = 'CPD2, with changing size')
plt.legend()
plt.xlabel('Where cpd1 starts')
plt.ylabel('Number of hits per')
plt.title('Testing if cpd 1 and 2 separation is working')


In [ ]:
def GetInterpFunc():
    """Creates an linear interpolation function from data found at the file path below,, giving us the ability to convert from resistance to temperature. 
    returns:
        Interpoltion function: If input exceeds range of the data function returns a NaN"""
    data = np.loadtxt('./dispersion_data.csv', delimiter=',')
    X = data[:,0]
    Y = data[:,1]
    return interp1d(X,Y, kind = 'linear')

In [ ]:
interp = GetInterpFunc()

In [ ]:
plt.subplot(211)
data = np.loadtxt('./dispersion_data.csv', delimiter=',')
v_data = np.loadtxt('./velocity_data.csv', delimiter=',')
X = data[:,0]
Y = data[:,1]
plt.plot(X, Y, label = 'Dispersion Curve')
plt.legend()
plt.show()
plt.subplot(212)
x = v_data[:,0]
y = v_data[:,1]
plt.plot(x[:-3], y[:-3], label = 'velocity')
plt.legend()
plt.show()

In [ ]:
def k_squared_distribution(u):
    N = (4.7**3 - .15**3)/3
    c = .15**3/(3 * N)
    return (3 * N* (u + c))**(1/3)

u = np.random.uniform(size=10000)
dist = k_squared_distribution(u)
plt.hist(dist, bins = 200)
plt.title('Distribution of K-values')
plt.xlabel('momenta')
plt.ylabel('counts/bin')